# LPatchTST — Kaggle Training Notebook

**Cell 1** – Clone repo from GitHub into `/kaggle/working/Lpatchtst` (handles nested dirs / submodules, symlinks Kaggle input data)  
**Cell 2** – Train on 2×T4 GPUs via `torchrun` (DDP)  
**Cell 3** – Evaluate using `batch_eval.py`

> **Before running:** Enable **Internet** and **GPU (T4 × 2)** in Notebook Settings.  
> Add your dataset under **Input** → it will appear at `/kaggle/input/<dataset-name>/`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone / pull repo + symlink Kaggle input data             ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys, glob

REPO_URL  = "https://github.com/ayan1-git/Lpatchtst"
REPO_DIR  = "/kaggle/working/Lpatchtst"
DATA_DST  = os.path.join(REPO_DIR, "Data")

# ── Point this at your Kaggle input dataset that contains the CSVs ─────
# e.g. if your dataset slug is  'ayan1-nifty-data'  the path will be:
#   /kaggle/input/ayan1-nifty-data/
# Leave as None to auto-discover the first input folder with CSVs.
DATA_SRC = None   # <- set explicitly if auto-discovery picks the wrong one

def run(cmd, cwd=None):
    """Stream a shell command; raise RuntimeError on non-zero exit."""
    print(f">>> {cmd}")
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True,
        cwd=cwd,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (rc={proc.returncode}): {cmd}")

# ── Clone or hard-reset to latest main ─────────────────────────────────
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — fetching latest…")
    run(f"git -C {REPO_DIR} fetch --recurse-submodules origin")
    run(f"git -C {REPO_DIR} reset --hard origin/main")
    run(f"git -C {REPO_DIR} submodule update --init --recursive")
else:
    print("Cloning repo (including submodules)…")
    run(f"git clone --recurse-submodules --depth 1 {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"CWD: {os.getcwd()}")

# ── Auto-discover Kaggle input dataset if DATA_SRC not set ─────────────
if DATA_SRC is None:
    for input_dir in sorted(glob.glob("/kaggle/input/*/")):
        if glob.glob(os.path.join(input_dir, "*.csv")):
            DATA_SRC = input_dir
            print(f"Auto-discovered data at: {DATA_SRC}")
            break
    if DATA_SRC is None:
        print("[WARN] No CSV found under /kaggle/input/ — set DATA_SRC manually.")

# ── Symlink input data into repo Data/ folder ───────────────────────────
if not os.path.exists(DATA_DST):
    if DATA_SRC and os.path.isdir(DATA_SRC):
        os.symlink(DATA_SRC.rstrip("/"), DATA_DST)
        print(f"Symlinked  {DATA_SRC}  →  {DATA_DST}")
    else:
        os.makedirs(DATA_DST, exist_ok=True)
        print(f"[WARN] Created empty {DATA_DST} — copy CSVs there before training.")
else:
    print(f"Data dir already present: {DATA_DST}")

# ── Install lightweight deps (torch already in Kaggle base image) ───────
run("pip install -q einops safetensors scikit-learn")

# ── Sanity: list detected CSVs ──────────────────────────────────────────
csvs = glob.glob(os.path.join(DATA_DST, '*.csv'))
print(f"\nCSVs in Data/: {len(csvs)}")
for c in csvs[:5]: print(f"  {os.path.basename(c)}")
if len(csvs) > 5: print(f"  … and {len(csvs)-5} more")

# ── GPU check ───────────────────────────────────────────────────────────
import torch
n_gpu = torch.cuda.device_count()
print(f"\nGPUs detected: {n_gpu}")
for i in range(n_gpu):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print("\n✅ Repo ready — proceed to Cell 2 to train.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Train on 2×T4 GPUs via torchrun (DDP)                     ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys, torch

REPO_DIR    = "/kaggle/working/Lpatchtst"
SCRIPT      = os.path.join(REPO_DIR, "train_pretrain_fine.py")
N_GPUS      = torch.cuda.device_count()   # auto: 2 on Kaggle T4×2 session
MASTER_PORT = 29500

if N_GPUS == 0:
    raise RuntimeError("No GPUs found — enable GPU in Kaggle session settings.")

os.chdir(REPO_DIR)   # must be CWD so config.py / Data/ resolve correctly

cmd = (
    f"torchrun "
    f"--standalone "
    f"--nnodes=1 "
    f"--nproc_per_node={N_GPUS} "
    f"--master_port={MASTER_PORT} "
    f"{SCRIPT}"
)

print(f"GPUs          : {N_GPUS} × {torch.cuda.get_device_name(0)}")
print(f"Command       : {cmd}")
print(f"Working dir   : {os.getcwd()}")
print("-" * 70)

# Stream stdout+stderr in real-time
proc = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
    cwd=REPO_DIR,
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
finally:
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"Training failed — torchrun exited with code {proc.returncode}")
print("\n✅ Training complete.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Evaluate (batch_eval.py, single process, no DDP)          ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys

REPO_DIR    = "/kaggle/working/Lpatchtst"
EVAL_SCRIPT = os.path.join(REPO_DIR, "audit_scripts", "batch_eval.py")

os.chdir(REPO_DIR)

# Evaluation is single-process — plain python, no torchrun
cmd = f"{sys.executable} {EVAL_SCRIPT}"

print(f"Running: {cmd}")
print("-" * 70)

proc = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
    cwd=REPO_DIR,
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
finally:
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"Evaluation failed — exited with code {proc.returncode}")
print("\n✅ Evaluation complete.")